In [ ]:
import rasterio
import numpy as np
import os
import re
import pandas as pd
from datetime import datetime, timedelta

In [ ]:
def parse_filename_for_date(tif_name):
    """
    Example filename: MOD13A2.061__1_km_16_days_EVI_doy2019001_aid0001.tif
    We'll search for a substring like 'doy2019001' and convert that to '2019-01-01'.
    Adjust if your filename pattern is different.
    """
    pattern = r"doy(\d{4})(\d{3})"  # e.g. doy2019001 => year=2019, day=001
    match = re.search(pattern, tif_name)
    if match:
        year = int(match.group(1))
        doy = int(match.group(2))
        # Convert year + day-of-year to a real date
        dt = datetime(year, 1, 1) + timedelta(days=doy - 1)
        return dt.strftime("%Y-%m-%d")
    else:
        # If you can't parse the date, store None or a placeholder
        return None

In [ ]:
# -------------------------------------------------------------------------------
# MAIN SCRIPT
# -------------------------------------------------------------------------------
tif_directory = "./tif"
tif_files = [f for f in os.listdir(tif_directory) if f.endswith('.tif')]

ndvi_data_list = []

for tif_file in tif_files:
    with rasterio.open(os.path.join(tif_directory, tif_file)) as src:
        # Read the first band (NDVI values)
        ndvi_data = src.read(1)  # Assuming NDVI data is in band 1
        transform = src.transform
        
        # Mask out NoData
        ndvi_data = np.ma.masked_equal(ndvi_data, src.nodata)
        
        # Use the custom function to parse date from filename
        date_str = parse_filename_for_date(tif_file)
        
        # Append a tuple of (date, masked array, transform)
        ndvi_data_list.append((date_str, ndvi_data, transform))

ndvi_df_list = []

In [ ]:
for date_str, ndvi_data, transform in ndvi_data_list:
    # rows/cols where NDVI is valid
    rows, cols = np.where(~ndvi_data.mask)
    
    latitudes = []
    longitudes = []
    
    # Convert pixel indices (row, col) to actual lat/lon
    for row, col in zip(rows, cols):
        lon, lat = transform * (col, row)
        latitudes.append(lat)
        longitudes.append(lon)
    
    # Create a DataFrame with lat, lon, NDVI, date
    temp_df = pd.DataFrame({
        'date': [date_str]*len(latitudes),
        'latitude': latitudes,
        'longitude': longitudes,
        'NDVI': ndvi_data[rows, cols]
    })
    
    ndvi_df_list.append(temp_df)

In [ ]:
ndvi_combined_df = pd.concat(ndvi_df_list, ignore_index=True)

# Save to CSV
ndvi_combined_df.to_csv("./california_ndvi_2019_2023.csv", index=False)

print(f"NDVI data combined and saved. Total records: {len(ndvi_combined_df)}")

In [ ]:
ndvi_check = pd.read_csv("california_ndvi_2019_2023.csv", nrows=5)
print(ndvi_check)

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import os

In [ ]:
db_path = 'data.db'  # SQLite file name
conn = sqlite3.connect(db_path)
cur = conn.cursor()

# ---------------------------------------------------------------------------
# 1. IMPORT NDVI DATA (CALIFORNIA_NDVI_2019_2023.CSV) IN CHUNKS
# ---------------------------------------------------------------------------
ndvi_csv_path = 'california_ndvi_2019_2023.csv'
chunksize = 100_000

cur.execute('DROP TABLE IF EXISTS ndvi')
conn.commit()

first_chunk = True

ndvi_iter = pd.read_csv(
    ndvi_csv_path,
    chunksize=chunksize,
    parse_dates=['date'],  # name must match CSV
    dtype={'latitude': 'float32','longitude': 'float32','NDVI': 'float32'}
)

for chunk in ndvi_iter:
    if not pd.api.types.is_datetime64_any_dtype(chunk['date']):
        chunk['date'] = pd.to_datetime(chunk['date'], errors='coerce')
    
    # Convert date to YYYY-MM-DD string for SQLite
    chunk['date'] = chunk['date'].dt.strftime('%Y-%m-%d')
    
    # Round lat/long
    chunk['lat_round'] = chunk['latitude'].round(2)
    chunk['lon_round'] = chunk['longitude'].round(2)
    
    # Write to SQLite
    if first_chunk:
        chunk.to_sql('ndvi', conn, if_exists='replace', index=False)
        first_chunk = False
    else:
        chunk.to_sql('ndvi', conn, if_exists='append', index=False)

# Create an index for faster lookups
cur.execute('CREATE INDEX IF NOT EXISTS idx_ndvi ON ndvi(lat_round, lon_round, date)')
conn.commit()
print("NDVI data loaded into 'ndvi' table and indexed.")

In [ ]:
# ---------------------------------------------------------------------------
# 2. IMPORT FIRES DATA (FIRES_MERGED_CLEANED.CSV) IN CHUNKS
# ---------------------------------------------------------------------------
fires_csv_path = 'fires_merged_cleaned.csv'

cur.execute('DROP TABLE IF EXISTS fires')
conn.commit()

first_chunk = True

fires_iter = pd.read_csv(
    fires_csv_path,
    chunksize=chunksize,
    dtype={
        'latitude': 'float32',
        'longitude': 'float32',
        'brightness': 'float32',
        'frp': 'float32'
    }
)

for chunk in fires_iter:
    # Rename date column if needed
    if 'ACQ_DATE' in chunk.columns:
        chunk.rename(columns={'ACQ_DATE': 'acq_date'}, inplace=True)
    if 'acq_datetime' in chunk.columns:
        chunk.rename(columns={'acq_datetime': 'acq_date'}, inplace=True)
    
    # Ensure 'acq_date' column
    if 'acq_date' not in chunk.columns:
        chunk['acq_date'] = pd.NaT
    
    # Convert to datetime, then string
    chunk['acq_date'] = pd.to_datetime(chunk['acq_date'], errors='coerce')
    chunk['acq_date'] = chunk['acq_date'].dt.strftime('%Y-%m-%d')
    
    # Round coordinates
    chunk['lat_round'] = chunk['latitude'].round(2)
    chunk['lon_round'] = chunk['longitude'].round(2)
    
    # Convert version to string if it exists
    if 'version' in chunk.columns:
        chunk['version'] = chunk['version'].astype(str)
    
    if first_chunk:
        chunk.to_sql('fires', conn, if_exists='replace', index=False)
        first_chunk = False
    else:
        chunk.to_sql('fires', conn, if_exists='append', index=False)

# Create an index for faster queries
cur.execute('CREATE INDEX IF NOT EXISTS idx_fires ON fires(lat_round, lon_round, acq_date)')
conn.commit()
print("Fires data loaded into 'fires' table and indexed.")

In [ ]:
columns_df = pd.read_sql_query("PRAGMA table_info(fires);", conn)
print(columns_df)

In [ ]:
# Step 3. Execute the nearest-date query using window functions
query = '''
WITH fires_with_jd AS (
    SELECT
        rowid AS f_rowid,
        latitude,
        longitude,
        acq_date,
        brightness,
        frp,
        version,
        lat_round,
        lon_round,
        julianday(acq_date) AS jd_acq_date
    FROM fires
),
ndvi_with_jd AS (
    SELECT
        rowid AS n_rowid,
        lat_round,
        lon_round,
        date,
        NDVI,
        julianday(date) AS jd_date
    FROM ndvi
),
candidate_matches AS (
    SELECT
        f.*,
        n.n_rowid,
        n.NDVI,
        ABS(n.jd_date - f.jd_acq_date) AS diff_in_days
    FROM fires_with_jd f
    JOIN ndvi_with_jd n
        ON f.lat_round = n.lat_round
       AND f.lon_round = n.lon_round
    WHERE ABS(n.jd_date - f.jd_acq_date) <= 8
),
ranked_matches AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY f_rowid
            ORDER BY diff_in_days ASC
        ) AS rn
    FROM candidate_matches
)
SELECT
    latitude,
    longitude,
    acq_date,
    brightness,
    frp,
    version,
    lat_round,
    lon_round,
    NDVI
FROM ranked_matches
WHERE rn = 1;
'''

print("Running nearest-date join query with window functions (±8 days). Please wait...")
merged_df = pd.read_sql_query(query, conn)
print("Query complete. Merged data loaded into a DataFrame.")


In [ ]:
# 4. Fix Data Types & Fill NDVI if requested
merged_df = merged_df.astype({
    'latitude': 'float32',
    'longitude': 'float32',
    'brightness': 'float32',
    'frp': 'float32',
    'NDVI': 'float32'
})
if 'version' in merged_df.columns:
    merged_df['version'] = merged_df['version'].astype('category')

# Check how many NDVIs are still NaN
nan_count = merged_df['NDVI'].isna().sum()
total_count = len(merged_df)
print(f"NDVI is NaN in {nan_count} of {total_count} rows.")

# Fill NDVI with mean if desired
if nan_count > 0:
    fill_mean = True  # Toggle this if you want to fill with mean
    if fill_mean:
        ndvi_mean = merged_df['NDVI'].mean(skipna=True)
        merged_df['NDVI'].fillna(ndvi_mean, inplace=True)
        print(f"Filled NaN NDVI with mean value {ndvi_mean:.2f}.")


In [ ]:
# 5. Save final result as CSV or Parquet
row_count = len(merged_df)
if row_count < 10_000_000:
    merged_df.to_csv("merged_final.csv", index=False)
    print("Final merged dataset saved to merged_final.csv.")
else:
    merged_df.to_parquet("merged_final.parquet", index=False)
    print("Final merged dataset saved to merged_final.parquet.")

# Clean up
conn.close()
print("All done! Database connection closed.")


In [ ]:
import pandas as pd

# 1. Row count
conn = sqlite3.connect(db_path)
fires_count = pd.read_sql_query("SELECT COUNT(*) AS cnt FROM fires", conn)
print("fires table count:", fires_count['cnt'][0])

# 2. Date range, lat/lon range
fires_stats = pd.read_sql_query("""
    SELECT 
        MIN(acq_date) AS min_date,
        MAX(acq_date) AS max_date,
        MIN(lat_round) AS min_lat_round,
        MAX(lat_round) AS max_lat_round,
        MIN(lon_round) AS min_lon_round,
        MAX(lon_round) AS max_lon_round
    FROM fires
""", conn)
print("fires date/coord stats:")
print(fires_stats)


In [ ]:
# 1. Row count
ndvi_count = pd.read_sql_query("SELECT COUNT(*) AS cnt FROM ndvi", conn)
print("ndvi table count:", ndvi_count['cnt'][0])

# 2. Date range, lat/lon range
ndvi_stats = pd.read_sql_query("""
    SELECT 
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        MIN(lat_round) AS min_lat_round,
        MAX(lat_round) AS max_lat_round,
        MIN(lon_round) AS min_lon_round,
        MAX(lon_round) AS max_lon_round
    FROM ndvi
""", conn)
print("ndvi date/coord stats:")
print(ndvi_stats)


In [ ]:
fires_preview = pd.read_sql_query("SELECT * FROM fires LIMIT 5", conn)
print("Fires table preview:")
print(fires_preview)

In [ ]:
ndvi_preview = pd.read_sql_query("SELECT * FROM ndvi LIMIT 5", conn)
print("NDVI table preview:")
print(ndvi_preview)


In [ ]:
import pandas as pd

ndvi_check = pd.read_csv("merged_final.csv", nrows=5)
print(ndvi_check)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("merged_final.csv")

print(df.info())
print(df.describe())
print(df.head(10))


In [ ]:
df['NDVI'].hist(bins=50)
plt.title("Distribution of NDVI")
plt.xlabel("NDVI")
plt.ylabel("Count")
plt.show()


In [ ]:
plt.scatter(df['NDVI'], df['frp'], alpha=0.1)
plt.title("FRP vs NDVI")
plt.xlabel("NDVI")
plt.ylabel("FRP")
plt.show()


In [ ]:
df = df[df['NDVI'] < 5000]  # Or any threshold you think is reasonable


In [ ]:
df['NDVI_log'] = np.log1p(df['NDVI'])
df['FRP_log'] = np.log1p(df['frp'])


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df[['NDVI', 'frp']] = scaler.fit_transform(df[['NDVI', 'frp']])


In [ ]:
df['NDVI'].hist(bins=50)
plt.title("Distribution of NDVI")
plt.xlabel("NDVI")
plt.ylabel("Count")
plt.show()


In [ ]:
import zipfile
import os

zip_dir = "./era5_wind/"  # Change this if needed
extract_path = "./era5_wind/extracted/"  # Output folder

# Make sure the extraction directory exists
os.makedirs(extract_path, exist_ok=True)

# Loop through all ZIP files in the directory
for file in os.listdir(zip_dir):
    if file.endswith(".nc"):  # Your files have .nc extension but are actually ZIPs
        file_path = os.path.join(zip_dir, file)
        if zipfile.is_zipfile(file_path):
            with zipfile.ZipFile(file_path, "r") as zip_ref:
                zip_ref.extractall(extract_path)
                print(f"✅ Extracted: {file}")

print("✅ All ZIP files extracted!")


In [ ]:
import os

extracted_files = os.listdir(extract_path)
print("Extracted files:", extracted_files)


In [ ]:
import xarray as xr

ds = xr.open_dataset("./era5_wind/data.nc", engine="netcdf4")
print(ds)


In [ ]:
import xarray as xr

# Load dataset
ds = xr.open_dataset("./era5_wind/data.nc")

# Print available variables
print(ds.data_vars)


In [ ]:
ds = ds.rename({"10u": "wind_u", "10v": "wind_v"})

In [ ]:
ds["wind_speed"] = (ds["wind_u"]**2 + ds["wind_v"]**2) ** 0.5

In [ ]:
ds.to_netcdf("./era5_wind/california_era5_wind.nc")

In [ ]:
import xarray as xr
import pandas as pd

# Open the combined wind dataset
ds_wind = xr.open_dataset("./era5_wind/california_era5_wind.nc")
print(ds_wind)

# Check available data variables (you already did this)
print("Data variables:", list(ds_wind.data_vars))

# If your dataset contains multiple time steps per day, aggregate to daily averages:
ds_wind_daily = ds_wind.resample(time='1D').mean()

# Convert the daily dataset to a DataFrame and reset the index
df_wind = ds_wind_daily.to_dataframe().reset_index()

# Create a 'date' column (as a Python date)
df_wind['date'] = pd.to_datetime(df_wind['time']).dt.date

# Round lat and lon to 1 decimal place to match your fire data
df_wind['lat_rounded'] = df_wind['lat'].round(1)
df_wind['lon_rounded'] = df_wind['lon'].round(1)

# Keep only the needed columns: date, lat_rounded, lon_rounded, and wind_speed
df_wind_final = df_wind[['date', 'lat_rounded', 'lon_rounded', 'wind_speed']]
print("Wind DataFrame (first 5 rows):")
print(df_wind_final.head())


In [4]:
import pandas as pd

df_final = pd.read_csv("final.csv")

C:\Users\student\AppData\Local\Temp\ipykernel_17520\982305492.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_final = pd.read_csv("final.csv")


In [ ]:
df_final_with_wind = pd.merge(
    df_final,
    df_wind_final,  # this is your wind DataFrame from previous steps
    on=['date', 'lat_rounded', 'lon_rounded'],
    how='left'
)

print("Merged DataFrame shape:", df_final_with_wind.shape)
print("Non-NaN wind_speed count:", df_final_with_wind['wind_speed'].notnull().sum())


In [ ]:
print("Fire data date (sample):", sorted(df_final['date'].unique())[:5])
print("Wind data date (sample):", sorted(df_wind_final['date'].unique())[:5])

print("Fire data lat_rounded (sample):", sorted(df_final['lat_rounded'].unique())[:5])
print("Wind data lat_rounded (sample):", sorted(df_wind_final['lat_rounded'].unique())[:5])

print("Fire data lon_rounded (sample):", sorted(df_final['lon_rounded'].unique())[:5])
print("Wind data lon_rounded (sample):", sorted(df_wind_final['lon_rounded'].unique())[:5])


In [ ]:
print("Wind data lat range:", df_wind_final['lat_rounded'].min(), "-", df_wind_final['lat_rounded'].max())
print("Wind data lon range:", df_wind_final['lon_rounded'].min(), "-", df_wind_final['lon_rounded'].max())

In [ ]:
import xarray as xr
ds_wind = xr.open_dataset("./era5_wind/california_era5_wind.nc")
print("Wind data date range:", ds_wind.time.min().values, "to", ds_wind.time.max().values)


In [ ]:
files = [
    "./era5_wind/data.nc",
]

ds_wind = xr.open_mfdataset(files, combine='by_coords')
print("Combined wind data date range:", ds_wind.time.min().values, "to", ds_wind.time.max().values)


In [42]:
import xarray as xr

# List of NetCDF files for all years
nc_files = [
    "./era5_wind/data_2019.nc",
    "./era5_wind/data_2020.nc",
    "./era5_wind/data_2021.nc",
    "./era5_wind/data_2022.nc",
    "./era5_wind/data_2023.nc"
]

# Open and merge the datasets
ds_wind_all = xr.open_mfdataset(nc_files, combine="by_coords")

# Print the new date range
print("✅ Combined wind data date range:", ds_wind_all.time.min().values, "to", ds_wind_all.time.max().values)


✅ Combined wind data date range: 2019-01-01T00:00:00.000000000 to 2023-12-31T18:00:00.000000000


In [43]:
ds_wind_all = ds_wind_all.rename({"10u": "wind_u", "10v": "wind_v"})

In [44]:
ds_wind_all["wind_speed"] = (ds_wind_all["wind_u"]**2 + ds_wind_all["wind_v"]**2) ** 0.5

In [45]:
ds_wind_daily = ds_wind_all.resample(time="1D").mean()


In [46]:
ds_wind_daily.to_netcdf("./era5_wind/california_era5_wind_daily.nc")

In [1]:
import xarray as xr
import pandas as pd
import numpy as np

# Open the daily wind dataset
ds_wind = xr.open_dataset("./era5_wind/california_era5_wind_daily.nc")
print("Wind dataset:")
print(ds_wind)

# Check available data variables
print("Wind data variables:", list(ds_wind.data_vars))

# Assuming your file already has a daily wind_speed variable, otherwise compute it:
if "wind_speed" not in ds_wind.data_vars:
    # If u10 and v10 exist, compute wind_speed:
    ds_wind["wind_speed"] = np.sqrt(ds_wind["u10"]**2 + ds_wind["v10"]**2)

# Convert the dataset to a DataFrame
df_wind = ds_wind.to_dataframe().reset_index()

# Create a 'date' column from the time dimension (as a Python date)
df_wind["date"] = pd.to_datetime(df_wind["time"]).dt.date

# Round latitude and longitude to 1 decimal place
df_wind["lat_rounded"] = df_wind["lat"].round(1)
df_wind["lon_rounded"] = df_wind["lon"].round(1)

# Keep only the columns needed for merging: date, lat_rounded, lon_rounded, wind_speed
df_wind_final = df_wind[["date", "lat_rounded", "lon_rounded", "wind_speed"]]
print("Processed wind DataFrame (first 5 rows):")
print(df_wind_final.head())


Wind dataset:
<xarray.Dataset> Size: 224MB
Dimensions:     (time: 1826, lat: 101, lon: 101)
Coordinates:
  * lon         (lon) float64 808B -124.0 -123.9 -123.8 ... -114.2 -114.1 -114.0
  * lat         (lat) float64 808B 42.0 41.9 41.8 41.7 ... 32.3 32.2 32.1 32.0
  * time        (time) datetime64[ns] 15kB 2019-01-01 2019-01-02 ... 2023-12-31
Data variables:
    wind_u      (time, lat, lon) float32 75MB ...
    wind_v      (time, lat, lon) float32 75MB ...
    wind_speed  (time, lat, lon) float32 75MB ...
Attributes:
    CDI:          Climate Data Interface version 2.4.0 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    institution:  European Centre for Medium-Range Weather Forecasts
    history:      Thu Feb 13 03:20:57 2025: cdo -f nc copy ./era5_wind/extrac...
    CDO:          Climate Data Operators version 2.4.0 (https://mpimet.mpg.de...
Wind data variables: ['wind_u', 'wind_v', 'wind_speed']
Processed wind DataFrame (first 5 rows):
         date  lat_rounded  lon_rounded  wi

In [5]:
# If df_final does not yet have a 'date' column, create it from 'acq_date'
if "date" not in df_final.columns:
    df_final["date"] = pd.to_datetime(df_final["acq_date"]).dt.date

# Merge on 'date', 'lat_rounded', and 'lon_rounded'
df_final_with_wind = pd.merge(
    df_final,
    df_wind_final,
    on=["date", "lat_rounded", "lon_rounded"],
    how="left"
)

print("Merged DataFrame shape:", df_final_with_wind.shape)
print("Non-NaN wind_speed count:", df_final_with_wind["wind_speed"].notnull().sum())


Merged DataFrame shape: (1190114, 22)
Non-NaN wind_speed count: 0


In [6]:
df_final_with_wind.to_csv("merged_with_wind.csv", index=False)
print("Dataset saved as merged_with_wind.csv")

Dataset saved as merged_with_wind.csv


In [7]:
print("Wind DataFrame wind_speed stats:")
print(df_wind_final["wind_speed"].describe())
print(df_wind_final["wind_speed"].head(10))


Wind DataFrame wind_speed stats:
count    1.397255e+07
mean     2.156938e+00
std      1.060583e+00
min      9.158154e-02
25%      1.423748e+00
50%      1.918497e+00
75%      2.631006e+00
max      1.674235e+01
Name: wind_speed, dtype: float64
0    2.265905
1    2.001145
2    1.807867
3    1.677509
4    1.556485
5    1.430776
6    1.298498
7    1.183173
8    1.129220
9    1.102778
Name: wind_speed, dtype: float32


In [8]:
import pandas as pd

# If df_final already has a wind_speed column, drop it.
if "wind_speed" in df_final.columns:
    df_final = df_final.drop(columns=["wind_speed"])

# Create a consistent date key in both datasets.
df_final["date_str"] = pd.to_datetime(df_final["acq_date"]).dt.strftime("%Y-%m-%d")
df_wind_final["date_str"] = pd.to_datetime(df_wind_final["date"]).dt.strftime("%Y-%m-%d")

# Ensure coordinate keys match. 
# If your fire dataset uses 'lat_round' and 'lon_round', rename the wind keys to match.
df_wind_final = df_wind_final.rename(columns={"lat_rounded": "lat_round", "lon_rounded": "lon_round"})

# Check the key samples:
print("Fire data date_str sample:", sorted(df_final["date_str"].unique())[:5])
print("Wind data date_str sample:", sorted(df_wind_final["date_str"].unique())[:5])
print("Fire data lat_round sample:", sorted(df_final["lat_round"].unique())[:5])
print("Wind data lat_round sample:", sorted(df_wind_final["lat_round"].unique())[:5])


C:\Users\student\AppData\Local\Temp\ipykernel_17520\527819650.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_wind_final["date_str"] = pd.to_datetime(df_wind_final["date"]).dt.strftime("%Y-%m-%d")


Fire data date_str sample: ['2019-01-01', '2019-01-02', '2019-01-03', '2019-01-04', '2019-01-06']
Wind data date_str sample: ['2019-01-01', '2019-01-02', '2019-01-03', '2019-01-04', '2019-01-05']
Fire data lat_round sample: [np.float64(32.529998779296875), np.float64(32.540000915527344), np.float64(32.54999923706055), np.float64(32.560001373291016), np.float64(32.56999969482422)]
Wind data lat_round sample: [np.float64(32.0), np.float64(32.1), np.float64(32.2), np.float64(32.3), np.float64(32.4)]


In [9]:
print(sorted(df_wind_final["lat_round"].unique()))


[np.float64(32.0), np.float64(32.1), np.float64(32.2), np.float64(32.3), np.float64(32.4), np.float64(32.5), np.float64(32.6), np.float64(32.7), np.float64(32.8), np.float64(32.9), np.float64(33.0), np.float64(33.1), np.float64(33.2), np.float64(33.3), np.float64(33.4), np.float64(33.5), np.float64(33.6), np.float64(33.7), np.float64(33.8), np.float64(33.9), np.float64(34.0), np.float64(34.1), np.float64(34.2), np.float64(34.3), np.float64(34.4), np.float64(34.5), np.float64(34.6), np.float64(34.7), np.float64(34.8), np.float64(34.9), np.float64(35.0), np.float64(35.1), np.float64(35.2), np.float64(35.3), np.float64(35.4), np.float64(35.5), np.float64(35.6), np.float64(35.7), np.float64(35.8), np.float64(35.9), np.float64(36.0), np.float64(36.1), np.float64(36.2), np.float64(36.3), np.float64(36.4), np.float64(36.5), np.float64(36.6), np.float64(36.7), np.float64(36.8), np.float64(36.9), np.float64(37.0), np.float64(37.1), np.float64(37.2), np.float64(37.3), np.float64(37.4), np.float6

In [10]:
print("Wind data date range:", df_wind_final["date"].min(), "to", df_wind_final["date"].max())


Wind data date range: 2019-01-01 to 2023-12-31


In [11]:
import pandas as pd

# For your fire dataset (df_final)
df_final["date_str"] = pd.to_datetime(df_final["acq_date"]).dt.strftime("%Y-%m-%d")
# Make sure fire data has its rounded coordinates; if needed, create them:
df_final["lat_round"] = df_final["lat_round"].round(1)  # or use your existing column if it's already rounded
df_final["lon_round"] = df_final["lon_round"].round(1)

# For your wind dataset (df_wind_final)
df_wind_final["date_str"] = pd.to_datetime(df_wind_final["date"]).dt.strftime("%Y-%m-%d")
# Rename wind data's coordinate columns to match fire data keys
df_wind_final = df_wind_final.rename(columns={"lat_rounded": "lat_round", "lon_rounded": "lon_round"})

# Quick check of keys
print("Fire data sample date_str:", sorted(df_final["date_str"].unique())[:5])
print("Wind data sample date_str:", sorted(df_wind_final["date_str"].unique())[:5])
print("Fire data lat_round sample:", sorted(df_final["lat_round"].unique())[:5])
print("Wind data lat_round sample:", sorted(df_wind_final["lat_round"].unique())[:5])


Fire data sample date_str: ['2019-01-01', '2019-01-02', '2019-01-03', '2019-01-04', '2019-01-06']
Wind data sample date_str: ['2019-01-01', '2019-01-02', '2019-01-03', '2019-01-04', '2019-01-05']
Fire data lat_round sample: [np.float64(32.5), np.float64(32.6), np.float64(32.7), np.float64(32.8), np.float64(32.9)]
Wind data lat_round sample: [np.float64(32.0), np.float64(32.1), np.float64(32.2), np.float64(32.3), np.float64(32.4)]


In [12]:
df_final_with_wind = pd.merge(
    df_final,
    df_wind_final[["date_str", "lat_round", "lon_round", "wind_speed"]],
    on=["date_str", "lat_round", "lon_round"],
    how="left"
)

print("Merged DataFrame shape:", df_final_with_wind.shape)
print("Non-NaN wind_speed count:", df_final_with_wind["wind_speed"].notnull().sum())


Merged DataFrame shape: (1190114, 23)
Non-NaN wind_speed count: 1189342


In [13]:
print(df_final_with_wind[["date_str", "lat_round", "lon_round", "wind_speed"]].head(10))


     date_str  lat_round  lon_round  wind_speed
0  2019-01-01       40.7     -121.8    2.601210
1  2019-01-01       39.4     -121.1    3.033852
2  2019-01-01       38.5     -120.7    2.418501
3  2019-01-01       38.3     -120.8    2.112766
4  2019-01-01       38.1     -122.1    4.849938
5  2019-01-01       38.0     -122.1    4.656133
6  2019-01-01       38.0     -122.4         NaN
7  2019-01-01       37.8     -121.7    3.137998
8  2019-01-01       36.4     -114.9    4.836262
9  2019-01-01       37.3     -120.3    1.263951


In [14]:
import numpy as np

# Recreate time-based features from acq_date
df_final_with_wind["acq_date"] = pd.to_datetime(df_final_with_wind["acq_date"], errors="coerce")
df_final_with_wind["day_of_year"] = df_final_with_wind["acq_date"].dt.dayofyear
df_final_with_wind["month"] = df_final_with_wind["acq_date"].dt.month
df_final_with_wind["year"] = df_final_with_wind["acq_date"].dt.year

# Create target variable (log-transformed FRP, for example)
df_final_with_wind["log_frp"] = np.log1p(df_final_with_wind["frp"])

# Define feature set, now including wind_speed
features = [
    "latitude", "longitude", "NDVI",
    "day_of_year", "month", "year",
    "temp_c", "dewpoint", "solar_radiation", "precip_mm",
    "wind_speed"
]

X = df_final_with_wind[features]
y = df_final_with_wind["log_frp"]

print("Features used:", features)
print(X.head())


Features used: ['latitude', 'longitude', 'NDVI', 'day_of_year', 'month', 'year', 'temp_c', 'dewpoint', 'solar_radiation', 'precip_mm', 'wind_speed']
   latitude  longitude    NDVI  day_of_year  month  year    temp_c   dewpoint  \
0  40.70188 -121.84412  0.0253            1      1  2019 -3.460846  259.02676   
1  39.38981 -121.12111  0.0124            1      1  2019 -0.179596  261.09512   
2  38.50407 -120.68475  0.0200            1      1  2019  0.234467  262.07950   
3  38.32057 -120.79156  0.0286            1      1  2019 -0.458893  266.05410   
4  38.07141 -122.13904  0.0211            1      1  2019  5.257904  266.24942   

   solar_radiation  precip_mm  wind_speed  
0         118126.0        0.0    2.601210  
1         127042.0        0.0    3.033852  
2         135592.0        0.0    2.418501  
3         141862.0        0.0    2.112766  
4         178098.0        0.0    4.849938  


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Use your tuned parameters from before
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=1,
    random_state=42
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("After adding wind speed:")
print("RMSE:", rmse)
print("R^2:", r2)

# Feature importances:
for feat, imp in zip(features, rf.feature_importances_):
    print(f"{feat}: {imp:.3f}")


After adding wind speed:
RMSE: 1.0070823210112527
R^2: 0.4111007593193834
latitude: 0.290
longitude: 0.271
NDVI: 0.082
day_of_year: 0.070
month: 0.002
year: 0.009
temp_c: 0.064
dewpoint: 0.066
solar_radiation: 0.060
precip_mm: 0.018
wind_speed: 0.068


In [22]:
import xarray as xr
import pandas as pd
import numpy as np

# ----------------------------
# Step 1: Process the Wind Data
# ----------------------------
# Open the daily wind dataset
ds_wind = xr.open_dataset("./era5_wind/california_era5_wind_daily.nc")
print("Wind dataset:")
print(ds_wind)

# If wind_direction is not already computed, compute it from wind_u and wind_v.
# Wind direction is computed as the angle (in degrees) from which the wind is coming.
if "wind_direction" not in ds_wind.data_vars:
    ds_wind["wind_direction"] = (np.degrees(np.arctan2(ds_wind["wind_u"], ds_wind["wind_v"])) + 360) % 360

# (Optional) Resample to daily averages if needed (here we assume daily data, but we use resample for safety)
ds_wind_daily = ds_wind.resample(time="1D").mean()

# Convert the daily wind data to a DataFrame
df_wind = ds_wind_daily.to_dataframe().reset_index()

# Create a date column (as a Python date) and a string version for merging
df_wind["date"] = pd.to_datetime(df_wind["time"]).dt.date
df_wind["date_str"] = pd.to_datetime(df_wind["date"]).dt.strftime("%Y-%m-%d")

# Round latitude and longitude to 1 decimal place to match your fire data keys
df_wind["lat_round"] = df_wind["lat"].round(1)
df_wind["lon_round"] = df_wind["lon"].round(1)

# Keep only the columns needed for merging: date_str, lat_round, lon_round, wind_speed, wind_direction
df_wind_final = df_wind[["date_str", "lat_round", "lon_round", "wind_speed", "wind_direction"]]
print("Wind DataFrame (first 5 rows):")
print(df_wind_final.head())

# ----------------------------
# Step 2: Prepare the Fire/NDVI Data for Merging
# ----------------------------
# Assume you have your fire/NDVI DataFrame loaded as df_final.
# df_final should contain an 'acq_date' column and already have rounded coordinate columns.
# If not already present, create them. For example:
if "date_str" not in df_final.columns:
    df_final["date_str"] = pd.to_datetime(df_final["acq_date"]).dt.strftime("%Y-%m-%d")

# Ensure your fire dataset has keys for merging.
# Here, we assume your fire dataset already contains 'lat_round' and 'lon_round' (rounded to 1 decimal).
print("Fire data (sample):")
print(df_final[["acq_date", "date_str", "lat_round", "lon_round"]].head())

# ----------------------------
# Step 3: Merge the Wind Data with the Fire Data
# ----------------------------
df_final_with_wind = pd.merge(
    df_final,
    df_wind_final,
    on=["date_str", "lat_round", "lon_round"],
    how="left"
)

print("Merged DataFrame shape:", df_final_with_wind.shape)
print("Non-NaN wind_speed count:", df_final_with_wind["wind_speed"].notnull().sum())
print("Non-NaN wind_direction count:", df_final_with_wind["wind_direction"].notnull().sum())

# Optionally, save the merged dataset for future use:
df_final_with_wind.to_csv("merged_with_wind_and_direction.csv", index=False)
print("Merged dataset saved as merged_with_wind_and_direction.csv")



Wind dataset:
<xarray.Dataset> Size: 224MB
Dimensions:     (time: 1826, lat: 101, lon: 101)
Coordinates:
  * lon         (lon) float64 808B -124.0 -123.9 -123.8 ... -114.2 -114.1 -114.0
  * lat         (lat) float64 808B 42.0 41.9 41.8 41.7 ... 32.3 32.2 32.1 32.0
  * time        (time) datetime64[ns] 15kB 2019-01-01 2019-01-02 ... 2023-12-31
Data variables:
    wind_u      (time, lat, lon) float32 75MB ...
    wind_v      (time, lat, lon) float32 75MB ...
    wind_speed  (time, lat, lon) float32 75MB ...
Attributes:
    CDI:          Climate Data Interface version 2.4.0 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    institution:  European Centre for Medium-Range Weather Forecasts
    history:      Thu Feb 13 03:20:57 2025: cdo -f nc copy ./era5_wind/extrac...
    CDO:          Climate Data Operators version 2.4.0 (https://mpimet.mpg.de...
Wind DataFrame (first 5 rows):
     date_str  lat_round  lon_round  wind_speed  wind_direction
0  2019-01-01       42.0     -124.0    2.2659